# 🦙 Ollama Clinical Text Extraction - Interactive Notebook

This notebook demonstrates **local LLM-based clinical text extraction** using the Ollama generator with the comprehensive EchoReport Pydantic model. All processing happens locally on your machine.

## Features
- **Local Processing**: No data leaves your machine - perfect for sensitive medical data
- **Ollama Integration**: Uses locally hosted Ollama models for extraction
- **Clinical-Grade Schema**: EchoReport model with comprehensive cardiac parameters
- **Batch Processing**: Efficient processing of multiple reports
- **Cost-Free**: No API costs - only local compute resources

## Prerequisites
1. Ollama installed and running locally (`ollama serve`)
2. A suitable model downloaded (e.g., `ollama pull llama3.2:3b`)
3. CSV file with echo report text data
4. ExtraCTOps venv_ollama environment activated

## Use Cases
- Extract structured data from clinical echo reports privately
- Process medical texts without cloud dependencies
- Create structured datasets for clinical research
- Validate model performance on real medical text

## 1. Setup and Configuration

Import libraries and configure the extraction parameters.

In [2]:
# Standard library imports
import os
import sys
import asyncio
import json
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Any, Optional

# Data processing
import pandas as pd

# Add project root to Python path
project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# ExtraCTOps imports
from utils.ExtraCTOps_loops import ProcessingConfig, ExtraCTOpsProcessor
from the_pydantics.EchoReport import EchoReport

print("✅ All libraries imported successfully!")
print(f"📁 Project root: {project_root}")
print(f"🐍 Python version: {sys.version}")
print(f"📊 Pandas version: {pd.__version__}")

ModuleNotFoundError: No module named 'utils'

## 2. Configuration Parameters

Set up the extraction configuration optimized for Ollama local processing.

In [ ]:
# Configuration optimized for Ollama local processing
CONFIG = {
    # Data settings
    "TEXT_COLUMN": "echo_report_text",
    "UID_COLUMN": "patient_id",
    
    # Processing settings (conservative for local processing)
    "BATCH_SIZE": 3,  # Lower batch size for local processing
    "BACKUP_INTERVAL": 5,
    "MAX_RETRIES": 2,
    
    # LLM settings
    "TEMPERATURE": 0.1,  # Low temperature for medical accuracy
    "MAX_TOKENS": 3000,
    "OLLAMA_MODEL": "llama3.2:3b",  # Good balance of speed and quality
    
    # Medical extraction prompts
    "SYSTEM_MESSAGE": """You are an expert cardiologist and medical data extraction specialist. 
Extract structured echocardiogram information from clinical reports with high accuracy. 
Focus on cardiac anatomy, function, measurements, and pathology. 
Return only valid JSON that matches the provided schema exactly.""",
    
    "PRE_PROMPT": """Analyze this echocardiogram report and extract all relevant cardiac information. 
Include measurements with units, anatomical descriptions, functional assessments, and any abnormalities. 
Be precise with medical terminology and numerical values. Return valid JSON only."""
}

# Setup output directory
output_dir = project_root / "exports" / "ollama_extractions"
output_dir.mkdir(parents=True, exist_ok=True)
CONFIG["OUTPUT_DIR"] = output_dir

print("🦙 OLLAMA EXTRACTION CONFIGURATION")
print("=" * 40)
for key, value in CONFIG.items():
    if key not in ["SYSTEM_MESSAGE", "PRE_PROMPT"]:  # Skip long text
        print(f"   {key}: {value}")

print(f"\n📁 Output directory: {CONFIG['OUTPUT_DIR']}")
print(f"🫀 Using EchoReport model with {len(EchoReport.model_fields)} main sections")

## 3. Ollama Status Check

Verify that Ollama is running and the specified model is available.

In [ ]:
# Check Ollama status and available models
import subprocess

def check_ollama_status():
    """Check if Ollama is running and what models are available."""
    try:
        # Check if Ollama is running
        result = subprocess.run(['ollama', 'list'], 
                              capture_output=True, text=True, timeout=10)
        
        if result.returncode == 0:
            print("✅ Ollama is running!")
            print("\n📋 Available models:")
            lines = result.stdout.strip().split('\n')
            if len(lines) > 1:  # Skip header
                for line in lines[1:]:
                    if line.strip():
                        model_name = line.split()[0]
                        print(f"   • {model_name}")
                        
                # Check if our configured model is available
                model_names = [line.split()[0] for line in lines[1:] if line.strip()]
                if CONFIG["OLLAMA_MODEL"] in model_names:
                    print(f"\n✅ Configured model '{CONFIG['OLLAMA_MODEL']}' is available!")
                    return True
                else:
                    print(f"\n⚠️  Configured model '{CONFIG['OLLAMA_MODEL']}' not found!")
                    print(f"   You can download it with: ollama pull {CONFIG['OLLAMA_MODEL']}")
                    return False
            else:
                print("   No models found. Download a model with: ollama pull llama3.2:3b")
                return False
        else:
            print("❌ Ollama not responding. Is it running?")
            print("   Start Ollama with: ollama serve")
            return False
            
    except FileNotFoundError:
        print("❌ Ollama not found. Please install Ollama first.")
        return False
    except subprocess.TimeoutExpired:
        print("❌ Ollama command timed out. Check if Ollama is running properly.")
        return False
    except Exception as e:
        print(f"❌ Error checking Ollama: {e}")
        return False

# Check Ollama status
ollama_available = check_ollama_status()

if not ollama_available:
    print("\n🔧 TROUBLESHOOTING:")
    print("1. Install Ollama: https://ollama.ai/")
    print("2. Start Ollama: ollama serve")
    print("3. Download a model: ollama pull llama3.2:3b")

## 4. Create or Load Sample Data

Create sample echo report data for testing, or modify to load your own CSV file.

In [ ]:
def create_sample_echo_data():
    """Create realistic sample echo report data for testing."""
    sample_reports = [
        {
            "patient_id": "ECHO_001",
            "study_date": "2024-01-15",
            "echo_report_text": """
ECHOCARDIOGRAM REPORT

Patient: 45-year-old male
Indication: Chest pain, rule out cardiac cause

FINDINGS:
Left Ventricle: The left ventricle is normal in size. Left ventricular systolic function is normal with an estimated ejection fraction of 65%. No regional wall motion abnormalities. LV diastolic volume 110 mL, systolic volume 38 mL.

Right Ventricle: The right ventricle is normal in size and systolic function.

Atria: The left atrium is mildly dilated. Right atrium is normal in size.

Valves: 
- Mitral valve is structurally normal with mild regurgitation
- Tricuspid valve shows mild regurgitation with estimated PA pressure 25 mmHg
- Aortic valve is structurally normal, trileaflet, no stenosis or regurgitation
- Pulmonary valve is normal with trivial regurgitation

Aorta: Aortic root measures 32 mm, ascending aorta 28 mm. Left aortic arch.

No pericardial effusion. No evidence of pulmonary hypertension.

IMPRESSION: Normal left ventricular size and systolic function. Mild left atrial dilation. Mild mitral and tricuspid regurgitation.
            """
        },
        {
            "patient_id": "ECHO_002", 
            "study_date": "2024-01-16",
            "echo_report_text": """
ECHOCARDIOGRAM REPORT

Patient: 62-year-old female
Indication: Hypertension, assessment of cardiac function

FINDINGS:
Left Ventricle: Moderate left ventricular hypertrophy. Estimated ejection fraction 45%, mildly depressed systolic function. LV diastolic volume 145 mL, systolic volume 80 mL.

Right Ventricle: Normal right ventricular size and function.

Atria: Both atria are moderately dilated. Left atrial volume indexed 38 mL/m².

Valves:
- Mitral valve shows mild stenosis and moderate regurgitation
- Aortic valve has mild stenosis with peak gradient 35 mmHg, mean gradient 20 mmHg
- Tricuspid regurgitation is moderate with elevated PA pressure 45 mmHg
- Pulmonary valve is normal

Great Vessels: Aortic root 35 mm, ascending aorta 40 mm.

Moderate pulmonary hypertension present with interventricular septal flattening in systole.

IMPRESSION: Moderate LV hypertrophy with mild systolic dysfunction. Moderate pulmonary hypertension. Mild aortic stenosis, moderate mitral regurgitation.
            """
        },
        {
            "patient_id": "ECHO_003",
            "study_date": "2024-01-17", 
            "echo_report_text": """
PEDIATRIC ECHOCARDIOGRAM REPORT

Patient: 8-year-old male
Indication: Heart murmur

FINDINGS:
Left Ventricle: Normal left ventricular size and systolic function, EF 65%.

Right Ventricle: Mild right ventricular dilation with normal systolic function.

Atria: Normal atrial sizes.

Septal Defects: 
- Small perimembranous ventricular septal defect, 4 mm, with left-to-right shunt
- Peak gradient across VSD 65 mmHg
- No atrial septal defect

Valves: All valves are structurally normal and competent.

Great Vessels: Normal aortic arch, no coarctation. Patent ductus arteriosus is absent.

Mild elevation of right heart pressures secondary to VSD.

IMPRESSION: Small perimembranous VSD with left-to-right shunt. Mild RV dilation. Normal valves and great vessels.
            """
        }
    ]
    
    df = pd.DataFrame(sample_reports)
    sample_file = project_root / "data" / "sample_echo_reports_ollama.csv"
    sample_file.parent.mkdir(exist_ok=True)
    df.to_csv(sample_file, index=False)
    
    print(f"✅ Created sample data: {sample_file}")
    print(f"📊 Sample data shape: {df.shape}")
    print(f"📋 Columns: {list(df.columns)}")
    
    return str(sample_file), df

# Create sample data
sample_file_path, sample_df = create_sample_echo_data()

# Display sample data
print(f"\n📊 Sample Data Preview:")
print(sample_df[["patient_id", "study_date"]].to_string(index=False))

# Option to use your own data (uncomment and modify the path)
# input_file_path = "/path/to/your/echo_reports.csv"
# input_df = pd.read_csv(input_file_path)

# Set the input file to use
input_file_path = sample_file_path
CONFIG["INPUT_FILE"] = input_file_path

## 5. Run Ollama Extraction

Execute the local LLM extraction using Ollama. This processes all reports through your local model.

In [ ]:
async def run_ollama_extraction():
    """Run extraction using local Ollama model."""
    if not ollama_available:
        print("❌ Cannot proceed - Ollama is not available")
        return None
    
    print("🦙 Starting Local Ollama Extraction")
    print("=" * 40)
    print(f"   Model: {CONFIG['OLLAMA_MODEL']}")
    print(f"   Batch size: {CONFIG['BATCH_SIZE']}")
    print(f"   Temperature: {CONFIG['TEMPERATURE']}")
    print(f"   Records to process: {len(sample_df)}")
    print(f"   Processing locally - no data leaves your machine! 🔒")
    
    try:
        # Configure Ollama extraction
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        output_file = str(CONFIG["OUTPUT_DIR"] / f"echo_ollama_{timestamp}.xlsx")
        
        extraction_config = ProcessingConfig(
            input_file=CONFIG["INPUT_FILE"],
            text_column=CONFIG["TEXT_COLUMN"],
            uid_column=CONFIG["UID_COLUMN"],
            pydantic_model=EchoReport,
            generator_type="ollama",
            model_name=CONFIG["OLLAMA_MODEL"],
            experiment_label="ollama_echo_extraction",
            batch_size=CONFIG["BATCH_SIZE"],
            backup_interval=CONFIG["BACKUP_INTERVAL"],
            max_retries=CONFIG["MAX_RETRIES"],
            temperature=CONFIG["TEMPERATURE"],
            max_tokens=CONFIG["MAX_TOKENS"],
            system_message=CONFIG["SYSTEM_MESSAGE"],
            pre_prompt=CONFIG["PRE_PROMPT"],
            output_file=output_file
        )
        
        # Run extraction
        processor = ExtraCTOpsProcessor(extraction_config)
        await processor.process_batch(open_file=False)
        
        # Calculate results
        successful = len([r for r in processor.results if r.success])
        failed = len([r for r in processor.results if not r.success])
        avg_time = sum(r.execution_time for r in processor.results) / len(processor.results) if processor.results else 0
        
        print(f"\n✅ Ollama extraction completed!")
        print(f"   ✓ Successful: {successful}")
        print(f"   ✗ Failed: {failed}")
        print(f"   📊 Success rate: {(successful/len(processor.results)*100):.1f}%")
        print(f"   ⏱️ Average time: {avg_time:.2f}s per report")
        print(f"   📁 Output file: {output_file}")
        
        return processor, extraction_config
        
    except Exception as e:
        print(f"❌ Extraction failed: {e}")
        import traceback
        traceback.print_exc()
        return None, None

# Run the extraction
if ollama_available:
    print("🚀 Starting extraction process...")
    processor, extraction_config = await run_ollama_extraction()
else:
    print("⏸️ Skipping extraction - Ollama not available")
    processor, extraction_config = None, None

## 6. Analyze Results

Examine the extraction results and display key statistics and sample outputs.

In [ ]:
def analyze_ollama_results(processor, config):
    """Analyze and display extraction results."""
    if not processor or not processor.results:
        print("❌ No extraction results to analyze")
        return
    
    print("📊 OLLAMA EXTRACTION ANALYSIS")
    print("=" * 40)
    
    # Overall statistics
    total_results = len(processor.results)
    successful = len([r for r in processor.results if r.success])
    failed = len([r for r in processor.results if not r.success])
    avg_time = sum(r.execution_time for r in processor.results) / total_results
    
    print(f"📈 Overall Performance:")
    print(f"   Total reports processed: {total_results}")
    print(f"   Successful extractions: {successful}")
    print(f"   Failed extractions: {failed}")
    print(f"   Success rate: {(successful/total_results)*100:.1f}%")
    print(f"   Average processing time: {avg_time:.2f}s per report")
    
    # Show sample successful extraction
    successful_results = [r for r in processor.results if r.success and r.extracted_data]
    if successful_results:
        sample = successful_results[0]
        print(f"\n🔍 Sample Successful Extraction:")
        print(f"   Patient ID: {sample.uid}")
        print(f"   Processing time: {sample.execution_time:.2f}s")
        print(f"   Extracted fields: {len(sample.extracted_data)}")
        
        # Show some extracted fields
        if sample.extracted_data:
            print(f"   Sample extracted data:")
            field_count = 0
            for field_name, value in sample.extracted_data.items():
                if value and field_count < 5:  # Show first 5 non-empty fields
                    print(f"      {field_name}: {value}")
                    field_count += 1
            if len(sample.extracted_data) > 5:
                print(f"      ... and {len(sample.extracted_data) - 5} more fields")
    
    # Show failed extractions if any
    failed_results = [r for r in processor.results if not r.success]
    if failed_results:
        print(f"\n⚠️ Failed Extractions:")
        for failed in failed_results[:3]:  # Show first 3 failures
            print(f"   Patient ID: {failed.uid}")
            print(f"   Error: {failed.error}")
        if len(failed_results) > 3:
            print(f"   ... and {len(failed_results) - 3} more failures")
    
    # Load and preview output file
    if config and config.output_file:
        try:
            output_df = pd.read_excel(config.output_file)
            print(f"\n📁 Output File Analysis:")
            print(f"   File: {config.output_file}")
            print(f"   Shape: {output_df.shape}")
            print(f"   Columns: {len(output_df.columns)}")
            
            # Show extraction status distribution
            status_col = f"{config.experiment_label}_status"
            if status_col in output_df.columns:
                status_counts = output_df[status_col].value_counts()
                print(f"   Status distribution:")
                for status, count in status_counts.items():
                    print(f"      {status}: {count}")
                    
        except Exception as e:
            print(f"   ❌ Error loading output file: {e}")

# Analyze results if extraction was successful
if processor and extraction_config:
    analyze_ollama_results(processor, extraction_config)
else:
    print("⏸️ No results to analyze - extraction was skipped or failed")

## 7. Save Summary Report

Create a comprehensive summary report of the extraction session.

In [ ]:
def save_ollama_summary_report(processor, config):
    """Save a comprehensive summary report."""
    if not processor or not processor.results:
        print("❌ No results to save")
        return
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Calculate detailed statistics
    total_results = len(processor.results)
    successful = len([r for r in processor.results if r.success])
    failed = len([r for r in processor.results if not r.success])
    avg_time = sum(r.execution_time for r in processor.results) / total_results
    total_time = sum(r.execution_time for r in processor.results)
    
    # Create comprehensive summary
    summary_report = {
        "extraction_session": {
            "timestamp": timestamp,
            "generator": "ollama",
            "model": CONFIG["OLLAMA_MODEL"],
            "local_processing": True,
            "input_file": CONFIG["INPUT_FILE"],
            "output_file": config.output_file if config else None,
            "processing_statistics": {
                "total_reports": total_results,
                "successful_extractions": successful,
                "failed_extractions": failed,
                "success_rate_percent": (successful / total_results) * 100,
                "average_time_per_report_seconds": avg_time,
                "total_processing_time_seconds": total_time
            },
            "configuration": {
                "batch_size": CONFIG["BATCH_SIZE"],
                "temperature": CONFIG["TEMPERATURE"],
                "max_tokens": CONFIG["MAX_TOKENS"],
                "backup_interval": CONFIG["BACKUP_INTERVAL"],
                "max_retries": CONFIG["MAX_RETRIES"]
            }
        },
        "pydantic_model": {
            "name": "EchoReport",
            "total_possible_fields": len(EchoReport.model_fields),
            "field_categories": list(EchoReport.model_fields.keys())[:10]  # Sample fields
        },
        "privacy_notes": {
            "local_processing": "All data processed locally with Ollama",
            "data_privacy": "No data sent to external APIs",
            "model_location": "Local machine only"
        }
    }
    
    # Save summary as JSON
    summary_file = CONFIG["OUTPUT_DIR"] / f"ollama_extraction_summary_{timestamp}.json"
    with open(summary_file, 'w') as f:
        json.dump(summary_report, f, indent=2, default=str)
    
    print(f"📋 Summary Report Saved")
    print(f"   File: {summary_file}")
    print(f"   Success rate: {summary_report['extraction_session']['processing_statistics']['success_rate_percent']:.1f}%")
    print(f"   Total time: {summary_report['extraction_session']['processing_statistics']['total_processing_time_seconds']:.1f}s")
    
    return summary_report

# Save summary report if extraction was successful
if processor and extraction_config:
    summary_report = save_ollama_summary_report(processor, extraction_config)
    
    print(f"\n🎉 OLLAMA EXTRACTION COMPLETE!")
    print(f"📁 All files saved to: {CONFIG['OUTPUT_DIR']}")
    
    # List generated files
    output_files = list(CONFIG["OUTPUT_DIR"].glob("*"))
    print(f"📋 Generated files ({len(output_files)}):")
    for file_path in sorted(output_files):
        file_size = file_path.stat().st_size / 1024  # KB
        print(f"   📄 {file_path.name} ({file_size:.1f} KB)")
    
    print(f"\n✨ Local extraction complete - all data stayed on your machine! 🔒")
else:
    print("⏸️ No summary to save - extraction was not completed")

## 🎯 Summary and Next Steps

### ✅ What We Accomplished

1. **Local LLM Processing**: Used Ollama for completely local text extraction
2. **Privacy-First**: All medical data processed locally without external API calls
3. **Clinical Schema**: Extracted structured data using the comprehensive EchoReport model
4. **Batch Processing**: Efficiently processed multiple reports with error handling
5. **Comprehensive Output**: Generated Excel files and JSON summaries

### 📊 Key Benefits of Ollama Extraction

- **🔒 Data Privacy**: No data leaves your machine
- **💰 Cost-Free**: No API costs or usage limits
- **⚡ Speed**: Local processing can be very fast
- **🔧 Customizable**: Full control over model and parameters
- **📱 Offline**: Works without internet connection

### 🚀 Next Steps

1. **Scale Up**: Process larger datasets with your local Ollama setup
2. **Model Optimization**: Try different Ollama models (llama3:8b, llama3:70b)
3. **Performance Tuning**: Adjust batch size based on your hardware
4. **Integration**: Use extracted data for clinical research or analysis
5. **Comparison**: Run the OpenAI notebook to compare cloud vs local results

### 🔧 Performance Tips

- **Hardware**: More RAM and CPU cores = faster processing
- **Model Selection**: Larger models (8b, 70b) may provide better accuracy
- **Batch Size**: Increase for more powerful hardware
- **Temperature**: Lower values (0.0-0.2) for more consistent medical extraction

---

**Ready for private, local clinical data extraction! 🦙✨**